# NB10 지분증명 — 에너지 대신 담보로

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB10-1: 담보 가중 무작위 선택 — 「코인 하나에 한 표」를 시뮬레이션으로 확인 (공부용, 시드 고정)
import random
rng = random.Random(20261025)

stakes = {"A": 32, "B": 32, "C": 64, "D": 128, "E": 256}      # 검증자별 담보(ETH 단위 장난감 값)
total = sum(stakes.values())
names, weights = list(stakes), list(stakes.values())

N = 100_000
picks = {n: 0 for n in names}
for _ in range(N):                                             # 슬롯마다 담보에 비례해 제안자 한 명을 뽑는다
    picks[rng.choices(names, weights=weights)[0]] += 1

print(f"{'검증자':>6} {'담보':>6} {'담보 비율':>9} {'선택 비율':>9}")
for n in names:
    print(f"{n:>6} {stakes[n]:>6} {stakes[n]/total:>9.3f} {picks[n]/N:>9.3f}")
print("담보가 두 배면 선택 횟수도 두 배 — 「코인 하나에 한 표」. 보상이 선택 횟수에 비례하면 담보에 비례한다.")
print("다만 유지되는 것은 비율이다 — 모두 같은 율로 늘면 절대 격차는 그 율만큼 커지고 비율은 그대로다. 부록 B ⑩은 이 점에서 「부분」 판정.")

   검증자     담보     담보 비율     선택 비율
     A     32     0.062     0.062
     B     32     0.062     0.061
     C     64     0.125     0.126
     D    128     0.250     0.253
     E    256     0.500     0.498
담보가 두 배면 선택 횟수도 두 배 — 「코인 하나에 한 표」. 보상이 선택 횟수에 비례하면 담보에 비례한다.
다만 유지되는 것은 비율이다 — 모두 같은 율로 늘면 절대 격차는 그 율만큼 커지고 비율은 그대로다. 부록 B ⑩은 이 점에서 「부분」 판정.


In [ ]:
# NB10-2: 슬래싱 벌금 — 현행 명세(Electra 이후) 규칙으로 즉시 벌금과 상관 벌금을 계산 (명세의 정수 나눗셈 순서 그대로)
GWEI = 10**9
MIN_SLASHING_PENALTY_QUOTIENT_ELECTRA = 4096
WHISTLEBLOWER_REWARD_QUOTIENT_ELECTRA = 4096
PROPORTIONAL_SLASHING_MULTIPLIER_BELLATRIX = 3
EFFECTIVE_BALANCE_INCREMENT = 1 * GWEI                        # 1 ETH = 10^9 Gwei (presets/mainnet/phase0.yaml)

def immediate_penalty(effective_balance_gwei: int) -> int:     # slash_validator
    return effective_balance_gwei // MIN_SLASHING_PENALTY_QUOTIENT_ELECTRA

def correlation_penalty(effective_balance_gwei: int, slashed_sum_gwei: int, total_active_gwei: int) -> int:   # process_slashings
    increment = EFFECTIVE_BALANCE_INCREMENT
    adjusted = min(slashed_sum_gwei * PROPORTIONAL_SLASHING_MULTIPLIER_BELLATRIX, total_active_gwei)
    per_increment = adjusted // (total_active_gwei // increment)
    return (effective_balance_gwei // increment) * per_increment

eb = 32 * GWEI                                                 # 32 ETH 검증자 하나
total_active = 1_000_000 * 32 * GWEI                           # 가정: 32 ETH 검증자 100만 명 = 3,200만 ETH 활성 (장난감 가정값)
print(f"즉시 벌금: {immediate_penalty(eb)/GWEI:.7f} ETH (= 32/4096), 신고자 보상: {eb // WHISTLEBLOWER_REWARD_QUOTIENT_ELECTRA / GWEI:.7f} ETH")
print(f"{'같은 기간 슬래싱된 담보 비율':>18} {'상관 벌금(ETH)':>14} {'잔고 대비':>8}")
for frac in [0.0001, 0.01, 0.10, 1/3, 0.5]:
    slashed_sum = int(total_active * frac)
    pen = correlation_penalty(eb, slashed_sum, total_active)
    print(f"{frac:>18.4%} {pen/GWEI:>14.4f} {pen/eb:>8.1%}")
print("혼자 어기면 둘째 벌금은 0에 가깝고, 총 담보의 1/3이 함께 어기면 3 × 1/3 = 100% — 전부를 잃는다.")

즉시 벌금: 0.0078125 ETH (= 32/4096), 신고자 보상: 0.0078125 ETH
  같은 기간 슬래싱된 담보 비율     상관 벌금(ETH)    잔고 대비
           0.0100%         0.0096     0.0%
           1.0000%         0.9600     3.0%
          10.0000%         9.6000    30.0%
          33.3333%        32.0000   100.0%
          50.0000%        32.0000   100.0%
혼자 어기면 둘째 벌금은 0에 가깝고, 총 담보의 1/3이 함께 어기면 3 × 1/3 = 100% — 전부를 잃는다.
